In [ ]:
import json

from pymongo import MongoClient

# Load data from JSONL file
client = MongoClient("mongodb://localhost:27017/")
db = client["ddxplus"]
collection_train = db["train-semistructured"]
collection_val = db["validate-semistructured"]
collection_test = db["test-semistructured"]

train_data = list(
    collection_train.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
val_data = list(
    collection_val.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
test_data = list(
    collection_test.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# show one train sample
print("Example train sample:")
print(json.dumps(train_data[0], indent=2))

TARGET_KEY = "DIFFERENTIAL_DIAGNOSIS_NOPROB"

In [ ]:
import random

import torch

from origami import DataConfig, ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.training import TableLogCallback, array_f1, array_jaccard

# For reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

config = OrigamiConfig(
    data=DataConfig(
        numeric_mode="none",
    ),
    model=ModelConfig(
        d_model=192,
        n_layers=6,
        n_heads=6,
    ),
    training=TrainingConfig(
        num_epochs=10,
        learning_rate=0.001,
        eval_strategy="steps",
        eval_steps=100,
        eval_sample_size=100,
        eval_metrics={"jaccard": array_jaccard, "f1": array_f1},
        shuffle_keys=True,
        upscale_factor=2,
        batch_size=100,
        target_key=TARGET_KEY,
    ),
)

pipeline = OrigamiPipeline(config)
callback = TableLogCallback(print_every=10)

In [ ]:
pipeline.fit(train_data, eval_data=val_data, epochs=50, callbacks=[callback], verbose=True)